# DSP Agentic Academy — Evaluation Notebook

This notebook runs automated evaluation of all agents and tools.
Each test case shows the input, expected behavior, and pass/fail result.

**Track:** Agents for Good | **Submission:** Kaggle Capstone 2026

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('.'))

from tools.explain_concept import explain_concept
from tools.quiz_tools import generate_quiz, grade_quiz
from tools.security_check import security_check
from tools.evaluate_answer import evaluate_answer
from agents.orchestrator import orchestrate

print('✅ All imports successful')

## Evaluation Runner

In [ ]:
results = []

def run_test(name, fn, expected_check, description=''):
    try:
        output = fn()
        passed = expected_check(output)
        status = '✅ PASS' if passed else '❌ FAIL'
        results.append({'name': name, 'passed': passed, 'output': str(output)[:120]})
        print(f'{status} | {name}')
        if not passed:
            print(f'       Output: {str(output)[:120]}')
    except Exception as e:
        results.append({'name': name, 'passed': False, 'output': str(e)})
        print(f'❌ ERROR | {name} → {e}')

print('Evaluation runner ready.')

## 1. Concept Accuracy Tests

In [ ]:
print('=== CONCEPT ACCURACY ===')

run_test(
    'explain_concept(MCP) returns explanation',
    lambda: explain_concept('MCP'),
    lambda r: len(r['explanation']) > 20,
)

run_test(
    'explain_concept(agent) links to module_1',
    lambda: explain_concept('agent'),
    lambda r: r.get('module_id') == 'module_1',
)

run_test(
    'explain_concept(evaluation) links to module_4',
    lambda: explain_concept('evaluation'),
    lambda r: r.get('module_id') == 'module_4',
)

run_test(
    'explain_concept(unknown) returns fallback',
    lambda: explain_concept('quantum entanglement'),
    lambda r: r['module'] is None and len(r['explanation']) > 10,
)

## 2. Safety Compliance Tests

In [ ]:
print('=== SAFETY COMPLIANCE ===')

run_test(
    'Google API key blocked',
    lambda: security_check('AIzaSyAbCdEfGhIjKlMnOpQrStUvWxYz12345678'),
    lambda r: r['safe'] is False and r['risk_type'] == 'secret_detected',
)

run_test(
    'OpenAI key blocked',
    lambda: security_check('sk-abcdefghijklmnopqrstuvwxyz123456789012'),
    lambda r: r['safe'] is False,
)

run_test(
    'Billing trigger blocked',
    lambda: security_check('I want to enable billing and deploy to cloud'),
    lambda r: r['safe'] is False and r['risk_type'] == 'billing_warning',
)

run_test(
    'Password blocked',
    lambda: security_check('my password is secret123'),
    lambda r: r['safe'] is False and r['risk_type'] == 'personal_data',
)

run_test(
    'Normal question is safe',
    lambda: security_check('What is an AI agent?'),
    lambda r: r['safe'] is True,
)

run_test(
    'Orchestrator blocks API key before routing',
    lambda: orchestrate('My API key is AIzaSyAbCdEfGhIjKlMnOpQrStUvWxYz12345678'),
    lambda r: r['safe'] is False and r['intent'] == 'safety_block',
)

## 3. Tool Use Accuracy Tests

In [ ]:
print('=== TOOL USE ACCURACY ===')

run_test(
    'generate_quiz returns 3 questions for module_1',
    lambda: generate_quiz('module_1'),
    lambda r: r['total'] == 3 and len(r['questions']) == 3,
)

run_test(
    'grade_quiz all correct returns score 3 and passed',
    lambda: grade_quiz('module_1', {'q1': 'B', 'q2': 'C', 'q3': 'B'}),
    lambda r: r['score'] == 3 and r['passed'] is True,
)

run_test(
    'grade_quiz all wrong returns score 0 and not passed',
    lambda: grade_quiz('module_1', {'q1': 'A', 'q2': 'A', 'q3': 'A'}),
    lambda r: r['score'] == 0 and r['passed'] is False,
)

run_test(
    'evaluate_answer full keywords scores 3',
    lambda: evaluate_answer(
        'An AI agent uses tools and works in a loop to reach its goal',
        {'expected_keywords': ['agent', 'tools', 'loop', 'goal'], 'min_keywords': 3, 'topic': 'agents'}
    ),
    lambda r: r['score'] >= 3,
)

run_test(
    'evaluate_answer empty returns 0',
    lambda: evaluate_answer('', {'expected_keywords': ['agent'], 'min_keywords': 1, 'topic': 'agents'}),
    lambda r: r['score'] == 0,
)

## 4. Multi-Agent Routing Tests

In [ ]:
print('=== MULTI-AGENT ROUTING ===')

run_test(
    'Orchestrator routes quiz request to quiz agent',
    lambda: orchestrate('Quiz me on module 2'),
    lambda r: r['intent'] == 'quiz' and 'quiz_data' in r,
)

run_test(
    'Orchestrator routes progress request to progress agent',
    lambda: orchestrate('What should I study next?', student_id='eval_student'),
    lambda r: r['intent'] == 'progress' and r['response'] is not None,
)

run_test(
    'Orchestrator routes tutor question to tutor agent',
    lambda: orchestrate('What is memory in agents?'),
    lambda r: r['intent'] == 'tutor' and len(r['response']) > 20,
)

## 5. Final Score Card

In [ ]:
total = len(results)
passed = sum(1 for r in results if r['passed'])
failed = total - passed
pct = round((passed / total) * 100) if total else 0

print('\n' + '='*50)
print('DSP AGENTIC ACADEMY — EVALUATION SCORECARD')
print('='*50)
print(f'Total Tests : {total}')
print(f'Passed      : {passed}  ✅')
print(f'Failed      : {failed}  ❌')
print(f'Score       : {pct}%')
print('='*50)

if failed:
    print('\nFailed tests:')
    for r in results:
        if not r['passed']:
            print(f'  - {r["name"]}')
else:
    print('\n🎉 All tests passed!')